# Python Parallel Programming

---

## 1. Thread vs Process in OS

A **process** is an independent program in execution with its own memory space, file descriptors, and OS resources. A **thread** is a lighter unit of execution that lives *inside* a process and shares its memory and resources with sibling threads.

| Feature | Process | Thread |
|---|---|---|
| Memory | Isolated (own address space) | Shared within process |
| Creation cost | High | Low |
| Communication | IPC (pipes, sockets, queues) | Shared memory (direct) |
| Crash isolation | Yes — one process crash doesn't affect others | No — a crash can kill all threads |
| Parallelism in Python | True (bypasses GIL) | Limited by GIL (for CPU work) |

---

## 2. Examples of Parallel Task Execution

- **Web server** handling multiple HTTP requests simultaneously.
- **Image processing pipeline** resizing thousands of photos across CPU cores.
- **Download manager** fetching multiple files concurrently over the network.
- **Data pipeline** reading from a database while another thread writes results to disk.
- **Game engine** running physics simulation, audio, and rendering on separate threads.
- **CI/CD system** running tests across multiple machines in parallel.

---

## 3. CPU-Bound vs IO-Bound Tasks

**CPU-bound** tasks spend most of their time doing computation. Adding more CPU cores speeds them up.

```python
# CPU-bound: heavy number crunching
def compute(n):
    return sum(i * i for i in range(n))
```

**IO-bound** tasks spend most of their time waiting for external resources (disk, network, database). Concurrency (not necessarily parallelism) speeds them up.

```python
# IO-bound: waiting for network response
import requests
def fetch(url):
    return requests.get(url).text
```

> **Rule of thumb:** Use `multiprocessing` for CPU-bound, use `threading` or `asyncio` for IO-bound.

---

## 4. GIL — Global Interpreter Lock

The **GIL** is a mutex in CPython that allows only **one thread to execute Python bytecode at a time**, even on multi-core machines.

### Why does it exist?

CPython manages memory via **reference counting**. Every object has a counter tracking how many variables point to it. When the count hits 0, the object is freed.

```python
import sys
x = []
print(sys.getrefcount(x))  # 2 (x + getrefcount argument)
```

Without the GIL, two threads could increment/decrement the same counter simultaneously, causing corruption or premature object deletion. The GIL serializes these operations, keeping reference counting safe without per-object locks.

### Consequence

- **IO-bound threads** release the GIL while waiting → threads work fine.
- **CPU-bound threads** hold the GIL while computing → no real parallelism.
- **Solution for CPU-bound:** use `multiprocessing` (separate processes = separate GILs).

> **Note:** Python 3.13 introduced an experimental "free-threaded" (no-GIL) mode via `--disable-gil` build flag.

---

## 5. Concurrency vs Parallelism

**Concurrency** means *dealing with* multiple tasks at once — tasks may interleave but don't necessarily run at the same physical instant.

**Parallelism** means *executing* multiple tasks at the exact same instant — requires multiple CPU cores.

```
Concurrency (single core):    [Task A]--[Task B]--[Task A]--[Task B]
Parallelism (multi-core):     Core 1: [Task A---------]
                              Core 2: [Task B---------]
```

In Python:
- `threading` / `asyncio` → concurrency (good for IO-bound).
- `multiprocessing` → parallelism (good for CPU-bound).

---

## 6. Race Conditions

A **race condition** occurs when the result depends on the unpredictable order in which threads access shared data.

```python
import threading

counter = 0

def increment():
    global counter
    for _ in range(100_000):
        counter += 1  # NOT atomic: read → add → write

t1 = threading.Thread(target=increment)
t2 = threading.Thread(target=increment)
t1.start(); t2.start()
t1.join(); t2.join()

print(counter)  # Expected 200000, but often less!
```

The fix is to use a **Lock** to make the critical section atomic (see next section).

---

## 7. Locks, Semaphores, and Other Synchronization Primitives

### Lock
Mutual exclusion — only one thread at a time.

```python
lock = threading.Lock()
with lock:
    counter += 1  # safe
```

### RLock (Reentrant Lock)
Same thread can acquire it multiple times without deadlocking.

```python
rlock = threading.RLock()
with rlock:
    with rlock:  # OK with RLock, would deadlock with Lock
        ...
```

### Semaphore
Allows up to *N* threads into a section simultaneously.

```python
sem = threading.Semaphore(3)  # max 3 concurrent DB connections

with sem:
    db.query(...)
```

### Event
One thread signals others to proceed.

```python
event = threading.Event()
# Thread 1
event.wait()  # blocks until set
# Thread 2
event.set()   # unblocks all waiters
```

### Condition
Combines a lock with wait/notify for producer-consumer patterns.

```python
cond = threading.Condition()
with cond:
    cond.wait()       # release lock and wait
    cond.notify_all() # wake waiters
```

### Barrier
All threads wait until *N* have reached the barrier, then proceed together.

```python
barrier = threading.Barrier(3)
barrier.wait()  # each thread blocks here until 3 arrive
```

---

## 8. Context Variables

`contextvars.ContextVar` provides **per-task / per-thread isolated state**, without using thread-local storage. It's especially designed for `asyncio` tasks where `threading.local()` doesn't help.

```python
from contextvars import ContextVar

request_id = ContextVar('request_id', default=None)

async def handle_request(rid):
    token = request_id.set(rid)
    await process()
    request_id.reset(token)

async def process():
    print(request_id.get())  # sees the correct per-task value
```

Each `asyncio` Task or thread automatically gets its own copy of the context. Useful for storing user session, locale, or trace IDs in web frameworks.

---

## 9. Web App Processes vs Threads (Gunicorn Workers)

When deploying a Python web app (e.g., Flask/Django), the worker type determines the concurrency model.

| Worker Type | Module | Model | Best For |
|---|---|---|---|
| `sync` (default) | built-in | 1 request per worker, blocking | Simple CPU-bound apps |
| `gthread` | built-in | Threaded workers | Mixed IO-bound workloads |
| `gevent` | `gevent` | Async greenlets (cooperative) | High-concurrency IO |
| `uvicorn.workers.UvicornWorker` | `uvicorn` | asyncio (ASGI) | Modern async apps |

```bash
# 4 sync worker processes
gunicorn -w 4 app:app

# 4 gthread workers, 2 threads each
gunicorn -w 4 --threads 2 -k gthread app:app

# Async gevent workers
gunicorn -w 4 -k gevent app:app
```

**Processes** give true isolation and parallelism (each has its own GIL) but use more RAM.  
**Threads** share memory (lower overhead) but are limited by the GIL for CPU work.

---

## 10. Cooperative vs Preemptive Multitasking

**Preemptive multitasking** (OS-level, threads): The OS can interrupt a running thread at any moment and switch to another. The thread has no say.

**Cooperative multitasking** (asyncio/gevent): A task runs until it *voluntarily* yields control (at `await`, `yield`, etc.). The event loop then picks the next task.

```python
# Cooperative — task yields at await
async def fetch(url):
    await asyncio.sleep(1)   # yields here; another task can run
    return "data"
```

| Aspect | Preemptive (threads) | Cooperative (asyncio) |
|---|---|---|
| Switching | OS decides, anytime | Task decides, at yield points |
| Race conditions | Possible anywhere | Only at yield points |
| Overhead | Context switch is costly | Very lightweight |
| Blocking call risk | Less (OS preempts) | High — one blocking call stalls all tasks |

---

## 11. Threads — `thread`, `threading`, `Queue`, Locks

### Low-level `_thread`
```python
import _thread
_thread.start_new_thread(print, ("hello",))
```

### `threading.Thread`
```python
import threading

def worker(n):
    print(f"Worker {n}")

t = threading.Thread(target=worker, args=(1,), daemon=True)
t.start()
t.join()  # wait for completion
```

### `threading.Queue` (thread-safe FIFO)
```python
from queue import Queue

q = Queue()
q.put("task")
item = q.get()
q.task_done()
q.join()  # blocks until all tasks done
```

### `ThreadPoolExecutor` (high-level)
```python
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=4) as pool:
    futures = [pool.submit(fetch, url) for url in urls]
    results = [f.result() for f in futures]
```

### Locks (recap in threading context)
```python
lock = threading.Lock()
with lock:
    shared_list.append(value)
```

---

## 12. Processes — `multiprocessing`

### `Process`
```python
from multiprocessing import Process

def work(x):
    print(x ** 2)

p = Process(target=work, args=(4,))
p.start()
p.join()
```

### `Queue` — inter-process communication
```python
from multiprocessing import Queue

q = Queue()
q.put(42)
print(q.get())  # works across processes
```

### `Pipe` — two-endpoint channel
```python
from multiprocessing import Pipe

parent_conn, child_conn = Pipe()
child_conn.send("hello")
print(parent_conn.recv())  # "hello"
```

### `Value` and `Array` — shared typed memory
```python
from multiprocessing import Value, Array

counter = Value('i', 0)   # 'i' = int
arr = Array('d', [1.0, 2.0, 3.0])  # 'd' = double

with counter.get_lock():
    counter.value += 1
```

### `Pool` — worker pool
```python
from multiprocessing import Pool

def square(x):
    return x * x

with Pool(4) as pool:
    results = pool.map(square, range(10))
```

### `ProcessPoolExecutor` (high-level, like ThreadPoolExecutor)
```python
from concurrent.futures import ProcessPoolExecutor

with ProcessPoolExecutor(max_workers=4) as pool:
    results = list(pool.map(square, range(10)))
```

### `Manager` — proxy objects for sharing complex data
```python
from multiprocessing import Manager

with Manager() as m:
    shared_dict = m.dict()
    shared_list = m.list()
    # Pass these to child processes
```

---

## 13. Shared Memory

`multiprocessing.shared_memory` (Python 3.8+) allows processes to share a raw memory block **without serialization**, making it much faster for large data like NumPy arrays.

```python
from multiprocessing import shared_memory
import numpy as np

# Process 1 — create shared memory
shm = shared_memory.SharedMemory(create=True, size=1024, name="my_block")
arr = np.ndarray((128,), dtype=np.float64, buffer=shm.buf)
arr[:] = np.arange(128)

# Process 2 — attach to existing block
shm2 = shared_memory.SharedMemory(name="my_block")
arr2 = np.ndarray((128,), dtype=np.float64, buffer=shm2.buf)
print(arr2[0])  # 0.0

# Cleanup
shm.close()
shm.unlink()  # delete from OS
```

`ShareableList` provides a list-like interface over shared memory:

```python
from multiprocessing.managers import SharedMemoryManager

with SharedMemoryManager() as smm:
    sl = smm.ShareableList([1, 2, 3, 4])
    print(sl[0])  # 1
```

> **Warning:** Shared memory has no built-in synchronization. Use `Lock` or `Semaphore` when multiple processes write to the same region.

---

## Quick Reference — Choosing the Right Tool

| Scenario | Recommended Tool |
|---|---|
| IO-bound, many tasks | `threading.ThreadPoolExecutor` or `asyncio` |
| CPU-bound computation | `multiprocessing.Pool` or `ProcessPoolExecutor` |
| Large data across processes | `shared_memory` + NumPy |
| Simple producer-consumer | `queue.Queue` (threads) or `multiprocessing.Queue` |
| Task isolation & crash safety | `multiprocessing.Process` |
| Per-request context in async app | `contextvars.ContextVar` |
| Limiting concurrent access | `threading.Semaphore` |